# ¿Cuánto cambia el error si la división es otra?

Respaldo de la pregunta de "Antes de la próxima sesión". Repite los dos mejores modelos del Ejercicio 2 con 10 divisiones distintas de `viajes_features.csv`; la primera es la división 42 que usa el registro.

Estas corridas **no se registran en MLflow** a propósito: si entraran en `tarifa-taxis-comparacion`, el registro tendría varias corridas de cada modelo sobre divisiones distintas, y `mejor_run_id` dejaría de comparar modelos sobre los mismos viajes.

In [1]:
import warnings

import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from preparacion_taxis import CATEGORICAS, NUMERICAS, OBJETIVO, YA_LISTAS, preparacion

warnings.filterwarnings("ignore")  # la red avisa si no converge; aqui solo importa su error

viajes = pd.read_csv("viajes_features.csv")
X = viajes[NUMERICAS + YA_LISTAS + CATEGORICAS]
y = viajes[OBJETIVO]
SEMILLAS = [42, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [2]:
def error_en_prueba(modelo, X_ent, X_prueba, y_ent, y_prueba):
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    pipeline.fit(X_ent, y_ent)
    return mean_absolute_error(y_prueba, pipeline.predict(X_prueba))


filas = []
for semilla in SEMILLAS:
    X_ent, X_prueba, y_ent, y_prueba = train_test_split(X, y, test_size=0.2, random_state=semilla)
    filas.append({
        "semilla_division": semilla,
        "gradient boosting": error_en_prueba(GradientBoostingRegressor(random_state=42), X_ent, X_prueba, y_ent, y_prueba),
        "red neuronal": error_en_prueba(MLPRegressor(hidden_layer_sizes=(64,), max_iter=1000, random_state=42), X_ent, X_prueba, y_ent, y_prueba),
    })

tabla = pd.DataFrame(filas)
tabla["gana la red"] = tabla["red neuronal"] < tabla["gradient boosting"]
print(tabla.round(4).to_string(index=False))

 semilla_division  gradient boosting  red neuronal  gana la red
               42             1.2754        1.1695         True
                1             1.1960        1.1626         True
                2             1.2143        1.2299        False
                3             1.0944        1.0954        False
                4             1.2377        1.2477        False
                5             1.2081        1.1860         True
                6             1.2234        1.2251        False
                7             1.1867        1.1690         True
                8             1.1996        1.1792         True
                9             1.2851        1.2333         True


In [3]:
modelos = ["red neuronal", "gradient boosting"]
print(tabla[modelos].agg(["mean", "std", "min", "max"]).round(4).T.to_string())
print()
print("La red gana en", int(tabla["gana la red"].sum()), "de", len(tabla), "divisiones")
print("Ventaja promedio de la red:", round(tabla["gradient boosting"].mean() - tabla["red neuronal"].mean(), 4))
print("Ventaja en la division 42: ", round(tabla.loc[0, "gradient boosting"] - tabla.loc[0, "red neuronal"], 4))

                     mean     std     min     max
red neuronal       1.1898  0.0456  1.0954  1.2477
gradient boosting  1.2121  0.0527  1.0944  1.2851

La red gana en 6 de 10 divisiones
Ventaja promedio de la red: 0.0223
Ventaja en la division 42:  0.1058
